In [1]:
import pandas as pd

df = pd.read_csv('movetimes3481887positions.csv')
df.head(10)

,fen,initial_time,increment,time_remaining_for_this_position,time_opponent_took,number_of_legal_moves,in_check,time_taken_for_this_position
0,rnbqkbnr/ppp1pppp/8/3P4/8/8/PPPP1PPP/RNBQKBNR ...,600,0,600.03,3.60,28,False,1.28
1,rnbqkbnr/pp2pppp/2p5/3P4/8/8/PPPP1PPP/RNBQKBNR...,600,0,596.43,1.28,31,False,3.52
2,rnbqkbnr/pp2pppp/2P5/8/8/8/PPPP1PPP/RNBQKBNR b...,600,0,598.75,3.52,32,False,1.44
3,rnbqkbnr/pp3ppp/2P5/4p3/8/8/PPPP1PPP/RNBQKBNR ...,600,0,592.91,1.44,31,False,3.28
4,rnbqkbnr/pP3ppp/8/4p3/8/8/PPPP1PPP/RNBQKBNR b ...,600,0,597.31,3.28,41,False,1.36
5,rn1qkbnr/pb3ppp/8/4p3/8/8/PPPP1PPP/RNBQKBNR w ...,600,0,589.63,1.36,29,False,6.32
6,rn1qkbnr/pb3ppp/8/4p3/8/2N5/PPPP1PPP/R1BQKBNR ...,600,0,595.95,6.32,43,False,1.44
7,rn1qk1nr/pb3ppp/8/2b1p3/8/2N5/PPPP1PPP/R1BQKBN...,600,0,583.31,1.44,32,False,10.16
8,rn1qk1nr/pb3ppp/8/2b1p3/1P6/2N5/P1PP1PPP/R1BQK...,600,0,594.51,10.16,47,False,2.00
9,rn1qk1nr/pb3ppp/1b6/4p3/1P6/2N5/P1PP1PPP/R1BQK...,600,0,573.15,2.00,33,False,13.76


In [2]:
from skrub import TableReport 
TableReport(df)

Processing column   8 / 8


,,,,,,,,


In [3]:
from sklearn.model_selection import train_test_split

nontest, test = train_test_split(df, test_size = 0.1, random_state = 42)

train, val = train_test_split(nontest, test_size = 0.1, random_state = 42)
# given the size of the datatest, cross-validation is unnecessary to maintain a large training dataset.

def drop_fen_column(data_frame):
    return data_frame.drop('fen', axis=1)

# dropping the fen column is done AFTER all the splitting...
# ...because train_test_split is sensitive to the data itself, even if the same random_state is used.
test = drop_fen_column(test)
train = drop_fen_column(train)
val = drop_fen_column(val)

In [4]:
train.info()

<class 'pandas.core.frame.DataFrame'>
Index: 2820328 entries, 744012 to 3336675
Data columns (total 7 columns):
 #   Column                            Dtype  
---  ------                            -----  
 0   initial_time                      int64  
 1   increment                         int64  
 2   time_remaining_for_this_position  float64
 3   time_opponent_took                float64
 4   number_of_legal_moves             int64  
 5   in_check                          bool   
 6   time_taken_for_this_position      float64
dtypes: bool(1), float64(3), int64(3)
memory usage: 153.3 MB


In [5]:
val.info()

<class 'pandas.core.frame.DataFrame'>
Index: 313370 entries, 489006 to 962524
Data columns (total 7 columns):
 #   Column                            Non-Null Count   Dtype  
---  ------                            --------------   -----  
 0   initial_time                      313370 non-null  int64  
 1   increment                         313370 non-null  int64  
 2   time_remaining_for_this_position  313370 non-null  float64
 3   time_opponent_took                313370 non-null  float64
 4   number_of_legal_moves             313370 non-null  int64  
 5   in_check                          313370 non-null  bool   
 6   time_taken_for_this_position      313370 non-null  float64
dtypes: bool(1), float64(3), int64(3)
memory usage: 17.0 MB


In [6]:
test.info()

<class 'pandas.core.frame.DataFrame'>
Index: 348189 entries, 3363431 to 298423
Data columns (total 7 columns):
 #   Column                            Non-Null Count   Dtype  
---  ------                            --------------   -----  
 0   initial_time                      348189 non-null  int64  
 1   increment                         348189 non-null  int64  
 2   time_remaining_for_this_position  348189 non-null  float64
 3   time_opponent_took                348189 non-null  float64
 4   number_of_legal_moves             348189 non-null  int64  
 5   in_check                          348189 non-null  bool   
 6   time_taken_for_this_position      348189 non-null  float64
dtypes: bool(1), float64(3), int64(3)
memory usage: 18.9 MB


In [7]:
from sklearn.dummy import DummyRegressor
from sklearn import linear_model

dummy_regression_mean = DummyRegressor(strategy="mean")
dummy_regression_median = DummyRegressor(strategy="median")
linear_regression = linear_model.LinearRegression()

# 'time_taken_for_this_position' is the target feature, so we'll take it out and put it in a variables called y
def X_y_split(dataframe):
    X = dataframe.drop('time_taken_for_this_position', axis=1)
    y = dataframe['time_taken_for_this_position']
    return (X, y)

train_X, train_y = X_y_split(train)
val_X, val_y = X_y_split(val)



In [8]:
from sklearn.metrics import mean_squared_error

dummy_regression_mean.fit(train_X, train_y)
dummy_regression_median.fit(train_X, train_y)
linear_regression.fit(train_X, train_y)

print('dummy_regression_mean score:', mean_squared_error(val_y, dummy_regression_mean.predict(val_X)))
print('dummy_regression_median score:', mean_squared_error(val_y, dummy_regression_median.predict(val_X)))
print('linear_regression score:', mean_squared_error(val_y, linear_regression.predict(val_X)))
# lower mean_squared_error is better

dummy_regression_mean score: 73.32519935919755
dummy_regression_median score: 80.64002901937006
linear_regression score: 57.240262667907466


In [9]:
print('features and their linear regression coefficients')
coefficients = linear_regression.coef_
index = 0
for column in train_X:
    print(column, coefficients[index])
    index += 1

features and their linear regression coefficients
initial_time 0.018332791765324426
increment 0.3421904005083799
time_remaining_for_this_position -0.010973219621055408
time_opponent_took 0.19932857605068743
number_of_legal_moves 0.1315660587322635
in_check 1.488935453116115


In [10]:
# NOTES ON LINEAR REGRESSION:
# time_remaining_for_this_position seems to be given a small coefficient (close to 0)
# This is against expectations: shouldn't it be a highly relevant feature? It has the highest Cramer V with the target feature.
# This is likely due to the difference in scale between features.